In [35]:
import torch
from torch import nn, optim

def forsaken(f_theta_0, T, lambda_, omega, P, D_f, eta_mu, xi):
    model_t = f_theta_0
    theta_0 = [param.clone().detach() for param in f_theta_0.parameters()]
    mu = [torch.zeros_like(param, requires_grad=True) for param in theta_0]
    criterion = nn.KLDivLoss()
    optimizer = optim.LBFGS(mu, lr=eta_mu)

    for _ in range(T):
        with torch.no_grad():
            for param, theta, m in zip(model_t.parameters(), theta_0, mu):
                param.copy_((theta - xi * m).detach())

        upsilon = model_t(D_f)
        loss = criterion(upsilon, P) + lambda_ * omega * sum(torch.norm(m, p=1) for m in mu)

        loss.backward()
        optimizer.step()

    return model_t



In [1]:
###################################
# 1) Imports
###################################
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader


C:\Users\mathi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\torch\utils\_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


In [3]:

###################################
# 2) Préparation des données MNIST
###################################
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(root="./data", train=True, transform=transform, download=True)
test_dataset = datasets.MNIST(root="./data", train=False, transform=transform, download=True)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)


100%|██████████| 9.91M/9.91M [00:01<00:00, 7.33MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 394kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 6.41MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 3.59MB/s]


In [88]:

###################################
# 3) Définition d'un modèle FC simple
###################################
class FullyConnectedNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(784, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.flatten(x)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

model = FullyConnectedNN()
model2 = FullyConnectedNN()

In [89]:

###################################
# 4) Entraînement rapide (optionnel)
###################################
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

epochs = 2  # juste pour illustrer
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for images, labels in train_loader:
        optimizer.zero_grad()
        output = model(images)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_loader):.4f}")



###################################
# 4) Entraînement rapide (optionnel)
###################################
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

epochs = 2  # juste pour illustrer
for epoch in range(epochs):
    model2.train()
    total_loss = 0
    for images, labels in train_loader:
        optimizer.zero_grad()
        output = model(images)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_loader):.4f}")



Epoch 1/2, Loss: 0.7806
Epoch 2/2, Loss: 0.3046
Epoch 1/2, Loss: 0.2478
Epoch 2/2, Loss: 0.2103


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

def forsaken(f_theta_0, T, lambda_, omega, P, D_f, eta_mu, xi):
    """
    f_theta_0 : Modèle cible (nn.Module) avec paramètres entraînables θ₀.
    T         : Nombre maximum d'itérations (entraînements).
    lambda_   : Poids de la pénalisation L1 sur μ.
    omega     : Hyperparamètre supplémentaire (dans la pénalisation).
    P         : Distribution cible (tensor) [batch_size, nb_classes].
    D_f       : Batch d'images à "oublier" (tensor) [batch_size, C, H, W].
    eta_mu    : Taux d'apprentissage (lr) pour l'optimizer SGD sur μ.
    xi        : Facteur d'influence : θ = θ₀ - xi * μ.
    """

    # Copie du modèle initial
    model_t = f_theta_0

    # On détache et clone les paramètres initiaux θ₀
    theta_0 = [param.clone().detach() for param in model_t.parameters()]

    # Vecteur mu initialisé à zéro (même forme que θ₀)
    mu = [torch.zeros_like(param, requires_grad=True) for param in theta_0]

    # Critère KLDivLoss (on utilisera log_softmax côté modèle)
    criterion_kl = nn.KLDivLoss(reduction='batchmean')

    # Optimiseur : SGD sur μ (pas sur les paramètres du modèle)
    optimizer_mu = optim.SGD(mu, lr=eta_mu)

    for _ in range(T):
        # 1) On remet à zéro les gradients de μ
        optimizer_mu.zero_grad()

        # 2) Mettre à jour les paramètres du modèle en fonction de μ
        #    θ_t = θ₀ - xi * μ_t
        for param, theta, m in zip(model_t.parameters(), theta_0, mu):
            # .data pour forcer la mise à jour in-place (on veut juste "imposer" θ)
            param.data = theta - xi * m

        # 3) Calcul de la prédiction du modèle
        #    Pour KLDivLoss, on donne en entrée des log-probabilités
        upsilon = F.log_softmax(model_t(D_f), dim=1)

        # 4) Calcul de la loss : KL + pénalisation sur μ
        loss = criterion_kl(upsilon, P) + lambda_ * omega * sum(torch.norm(m, p=1) for m in mu)

        # 5) Backward pour mettre à jour μ
        loss.backward()

        # 6) Descente SGD sur μ
        optimizer_mu.step()

    return model_t


In [90]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.nn.utils.stateless import functional_call

def forsaken(f_theta_0, T, lambda_, omega, P, D_f, eta_mu, xi):
    # On travaille sur le modèle cible
    model_t = f_theta_0
    # On crée un dictionnaire de paramètres initiaux détachés
    theta_0 = {name: param.clone().detach() for name, param in model_t.named_parameters()}
    # On initialise μ pour chaque paramètre (avec requires_grad=True)
    mu = {name: torch.zeros_like(param, requires_grad=True) for name, param in theta_0.items()}
    
    criterion = nn.KLDivLoss(reduction='batchmean')
    # Optimiseur LBFGS sur les valeurs de mu
    optimizer = optim.LBFGS(list(mu.values()), lr=eta_mu)
    
    def closure():
        optimizer.zero_grad()
        # Calculer les nouveaux paramètres de manière différentiable
        new_params = {name: theta_0[name] - xi * mu[name] for name in theta_0}
        # Effectuer la forward pass avec ces nouveaux paramètres
        upsilon = functional_call(model_t, new_params, D_f)
        log_probs = F.log_softmax(upsilon, dim=1)
        loss = criterion(log_probs, P) + lambda_ * omega * sum(torch.norm(m, p=1) for m in mu.values())
        loss.backward()
        print(f"Loss: {loss.item()}")
        return loss

    for _ in range(T):
        optimizer.step(closure)
    
    # Après optimisation, mettre à jour les paramètres du modèle
    final_params = {name: theta_0[name] - xi * mu[name].detach() for name in theta_0}
    for name, param in model_t.named_parameters():
        param.data.copy_(final_params[name])
    
    return model_t


In [91]:
import random
from torch.utils.data import Subset, DataLoader

# Sélectionner 100 indices aléatoires depuis train_dataset
forget_indices = random.sample(range(len(train_dataset)), 100)

# Créer le sous-ensemble D_f
D_f = Subset(train_dataset, forget_indices)
print(f"Nombre de points dans D_f à forget : {len(D_f)}")

# Optionnel : créer un DataLoader pour D_f (pour l'évaluation)
forget_loader = DataLoader(D_f, batch_size=64, shuffle=False)


Nombre de points dans D_f à forget : 100


In [92]:
import random
from torch.utils.data import Subset, DataLoader
import torch.nn.functional as F

# Supposons que train_dataset et model (votre modèle initial) soient déjà définis

# 1. Création d'un sous-ensemble D_f (par exemple, 100 points aléatoires)
forget_indices = random.sample(range(len(train_dataset)), 100)
D_f = Subset(train_dataset, forget_indices)
forget_loader = DataLoader(D_f, batch_size=64, shuffle=False)
print(f"Nombre de points dans D_f à forget : {len(D_f)}")

# 2. On prend un batch du DataLoader de D_f pour passer à forsaken
images_forget, labels_forget = next(iter(forget_loader))

# 3. Création de la distribution cible P (uniforme) pour chaque image du batch
P = torch.full((images_forget.size(0), 10), 1/10)

# 4. Définition des hyperparamètres pour forsaken
T = 10       # Nombre d'itérations externes
lambda_ = 1.0
omega = 1.0
eta_mu = 0.1
xi = 0.5

# 5. Appel de la fonction forsaken pour obtenir le modèle modifié
model_forsaken = forsaken(
    f_theta_0=model,
    T=T,
    lambda_=0,
    omega=omega,
    P=P,
    D_f=images_forget,
    eta_mu=eta_mu,
    xi=xi
)
print("Fin de la procédure Forsaken.")

# 6. Évaluation de model_forsaken sur l'ensemble D_f
def evaluate_model_on_forget(model, data_loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in data_loader:
            outputs = model(images)
            _, preds = torch.max(outputs, dim=1)
            total += labels.size(0)
            correct += (preds == labels).sum().item()
    return 100 * correct / total

accuracy_forget = evaluate_model_on_forget(model_forsaken, forget_loader)
print(f"Accuracy sur D_f après Forsaken : {accuracy_forget:.2f}%")

accuracy_base = evaluate_model_on_forget(model2, forget_loader)
print(f"Accuracy sur D_f après Forsaken : {accuracy_base:.2f}%")


Nombre de points dans D_f à forget : 100
Loss: 6.314843654632568
Loss: 6.313352584838867
Loss: 5.150413990020752
Loss: 4.209206581115723
Loss: 3.146730422973633
Loss: 2.2081868648529053
Loss: 1.5361204147338867
Loss: 1.1342324018478394
Loss: 0.8615572452545166
Loss: 0.6811269521713257
Loss: 0.5367258191108704
Loss: 0.4201909601688385


C:\Users\mathi\AppData\Local\Temp\ipykernel_8828\208682766.py:24: FutureWarning: `torch.nn.utils.stateless.functional_call` is deprecated as of PyTorch 2.0 and will be removed in a future version of PyTorch. Please use `torch.func.functional_call` instead which is a drop-in replacement.
  upsilon = functional_call(model_t, new_params, D_f)


Loss: 0.3368071913719177
Loss: 0.2732372581958771
Loss: 0.22145381569862366
Loss: 0.18298502266407013
Loss: 0.1537044197320938
Loss: 0.12938746809959412
Loss: 0.10884270817041397
Loss: 0.09220293909311295
Loss: 0.07926866412162781
Loss: 0.0686633437871933
Loss: 0.05762981250882149
Loss: 0.05040331557393074
Loss: 0.04358097165822983
Loss: 0.03807435184717178
Loss: 0.0332292839884758
Loss: 0.02914389595389366
Loss: 0.024684147909283638
Loss: 0.021226108074188232
Loss: 0.018635248765349388
Loss: 0.016296831890940666
Loss: 0.014046658761799335
Loss: 0.01213228702545166
Loss: 0.010323054157197475
Loss: 0.009063559584319592
Loss: 0.007944460958242416
Loss: 0.006953696254640818
Loss: 0.006052433513104916
Loss: 0.005278617609292269
Loss: 0.0046281348913908005
Loss: 0.004130208864808083
Loss: 0.0036650802940130234
Loss: 0.0032605198211967945
Loss: 0.00292453495785594
Loss: 0.0025890558026731014
Loss: 0.0023289253003895283
Loss: 0.002109148306772113
Loss: 0.0018917799461632967
Loss: 0.0017209725